Read Bronze

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("taxi_catalog.bronze.yellow_taxi_raw")

print("Bronze row count:", df_bronze.count())
display(df_bronze.limit(10))

null indetify 

In [0]:
df_bronze.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("tpep_pickup_datetime").isNull().cast("int")).alias("null_pickup"),
    F.sum(F.col("tpep_dropoff_datetime").isNull().cast("int")).alias("null_dropoff"),
    F.sum(F.col("trip_distance").isNull().cast("int")).alias("null_distance"),
    F.sum(F.col("fare_amount").isNull().cast("int")).alias("null_fare"),
    F.sum(F.col("total_amount").isNull().cast("int")).alias("null_total")
).show()

Invalid records ka profile

In [0]:
df_bronze.select(
    F.sum(
        (F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"))
        .cast("int")
    ).alias("dropoff_before_pickup"),

    F.sum(
        (F.col("trip_distance") < 0).cast("int")
    ).alias("negative_distance"),

    F.sum(
        (F.col("trip_distance") == 0).cast("int")
    ).alias("zero_distance"),

    F.sum(
        (F.col("total_amount") < 0).cast("int")
    ).alias("negative_total"),

    F.sum(
        (F.col("passenger_count") < 0).cast("int")
    ).alias("negative_passengers")
).show()

Trip duration calculate karke check karo

In [0]:
df_profile = df_bronze.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

df_profile.select(
    F.sum(
        (F.col("trip_duration_minutes") <= 0).cast("int")
    ).alias("zero_or_negative_duration"),

    F.sum(
        (F.col("trip_duration_minutes") > 1440).cast("int")
    ).alias("duration_over_24_hours")
).show()

Apply quality rules


In [0]:
df_silver = (
 df_bronze
 .filter(F.col("tpep_pickup_datetime").isNotNull())
 .filter(F.col("tpep_dropoff_datetime").isNotNull())
 .filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
 .filter(F.col("trip_distance").isNotNull())
 .filter(F.col("trip_distance") >= 0)
 .filter(F.col("fare_amount").isNotNull())
 .filter(F.col("total_amount").isNotNull())
)


Derive analytical columns

In [0]:
df_silver = (
 df_silver
 .withColumn(
 "trip_duration_minutes",
 (F.col("tpep_dropoff_datetime").cast("long")
 - F.col("tpep_pickup_datetime").cast("long")) / 60
 )
 .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
 .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
 .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
 .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
)
df_silver = df_silver.filter(
 (F.col("trip_duration_minutes") > 0) &
 (F.col("trip_duration_minutes") <= 24 * 60)
)


silver cleaning data frame 


In [0]:
df_silver = (
    df_profile
    .filter(
        (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")) &
        (F.col("trip_duration_minutes") <= 1440) &
        (F.col("trip_distance") >= 0) &
        F.col("fare_amount").isNotNull() &
        F.col("total_amount").isNotNull()
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn(
        "amount_type",
        F.when(F.col("total_amount") < 0, "Adjustment")
         .otherwise("Normal")
    )
)

print("Bronze rows:", df_bronze.count())
print("Silver rows:", df_silver.count())
print("Rows removed:", df_bronze.count() - df_silver.count())

display(df_silver.limit(10))

silver data schema create and save into table 

In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS taxi_catalog.silver")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_catalog.silver.yellow_taxi_clean")

Compare row counts

In [0]:
df_bronze_count = (spark.table("taxi_catalog.bronze.yellow_taxi_raw"))
silver_count = (spark.table("taxi_catalog.silver.yellow_taxi_clean").count())

print("Bronze rows:", df_bronze_count.count())
print("Silver rows:", silver_count)
print("Rows removed:", df_bronze_count.count() - silver_count)


Silver validation


In [0]:
df_silver_check = spark.table("taxi_catalog.silver.yellow_taxi_clean")

print("silver_rows:",df_silver_check.count())
print("silver_columns:",len(df_silver_check.columns))

display(df_silver_check.limit(10))